> `pip install uv && uv pip install --system -r https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/requirements.txt`

>`./DATA/pyare/Routine/LLM/Reasoning/LLMs-from-scratch-pyare/setup_venv.sh` following this as i don't want to keep .cache at home.

# 2. Working with text data

## 2.1 Understanding word embeddings

- NN can not process the raw text data since text is categorical data, it isn't compatible with the mathematical operations used to implement and train neural networks.
- Therefore, we need a way to represent words as continous-valued vectors.

- The concept of converting data into a vector format is often referred to as **embeddings**
- We embed different data types -  for example, 
    - Video
    - audio
    - text
    - point cloud
- However, its important to note that different data formats require distinct embedding models.

![Embedding models transforms the raw data into dense vector representation](/DATA/pyare/Routine/LLM/Reasoning/LLMs-from-scratch-pyare/chapter-2-tokenization/2.2_converting_raw_embedding_example.png)

- The primary purpose of embeddings is to convert nonnumeric data into a format that neural networks can process.
- While word embeddings are the most common form of text embeding, there are also embeddings for 
    - Sentences
    - Paragrahs
    - There are popular choices for *retrival-augmented generation* (RAG)
    - In this book focus is on *word embedding*

**Word Embedings**: The old traditional methods are *Word2Vec* : Idea is words that appears in similar contexts tend to have similar meanings. Consequently , when projected into two-dimensional word embeddings for visualization purposes, similar terms are clustered together.

![figure 2.3](/DATA/pyare/Routine/LLM/Reasoning/LLMs-from-scratch-pyare/chapter-2-tokenization/2.3.png)

- Word embedings can have varying dimensions, from one to thousands. A higher dimensionality might capture more nuasanced relationships but at the cost of computational efficiency.

-  GPT-2 (117 M and 125 M parameters ) uses an embedding size of 768 dimensions.
- the largest GPT-3 model ( 175 B parameters ) uses an embedding dimension of 12,288

- Next we will walk through **Spliting text into words**, **converting words into tokens** and then turning **tokens into embedding vectors**.

## 2.2 Tokenize Text

<div align ="center">
<img src ="/DATA/pyare/Routine/LLM/Reasoning/LLMs-from-scratch-pyare/chapter-2-tokenization/2.4.png" width="600">
</div>

The text we will tokenize for training is "The Verdict", a short story by Edith Wharton is released into the public domain and thus permitted to be used for LLM training tasks. The text is available on Wikisource at [https://en.wikisource.org/wiki/The_Verdict](https://en.wikisource.org/wiki/The_Verdict)

#### Download the dataset "The verdict"

In [ ]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)


# The book originally used the following code below
# However, urllib uses older protocol settings that
# can cause problems for some readers using a VPN.
# The `requests` version above is more robust
# in that regard.

"""
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)
"""

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

In [ ]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

In [ ]:
result = re.split(r'([,.]|\s)', text)

print(result)

In [ ]:
# Strip whitespace from each item and then filter out any empty strings.
result = [item for item in result if item.strip()]
print(result)

In [ ]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

In [ ]:
print(len(preprocessed))

In [ ]:
# 2.3 Converting the tokens into token IDS 

![2.5](/DATA/pyare/Routine/LLM/Reasoning/LLMs-from-scratch-pyare/chapter-2-tokenization/2.5.png)

- We build a vocabulary by tokenizing the entire text in a training dataset into individual tokens. These individual tokens are then sorted alphabetically, and duplicate tokens are removed.
- The unique tokens are then aggregated into a vocabulary that defines a mapping from each unique token to a unique integer value.

![2.7](/DATA/pyare/Routine/LLM/Reasoning/LLMs-from-scratch-pyare/chapter-2-tokenization/2.7.png)

- When we want to convert the outputs of an LLM from numbers back into text, we need a way to turn token IDs into text.
- For this, we can create an inverse version of the vocabulary that maps token IDs back to the corresponding text tokens.

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

In [ ]:
vocab = {token:integer for integer,token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

![2.8](/DATA/pyare/Routine/LLM/Reasoning/LLMs-from-scratch-pyare/chapter-2-tokenization/2.8.png)

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

- We can decode the integers back into text

In [ ]:
tokenizer.decode(ids)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

In [ ]:
text = "Hello , do you like tea ?"
print(tokenizer.encode(text))

- The problem is that word "Hello" is not used in the story "The verdict", this could be same as we never seen a point/superpoint/or cluster or reprentative points by the encoder.
- Hence it is not contained in the vocabulary
- **this highlights the large and diverse training sets to extend the vocabulary when working on LLMs.**

- Next we will test the tokenizer futher on text that contains unknown words and discuss additional special tokens that can be used to provide the further context for LLM during training.

### 2.4 Adding special context tokens
 -  Now we will modify the vocabulary and tokenizer to include the two new tokens `<|unk|>`,`<|endoftext|>`

<img src ="/DATA/pyare/Routine/LLM/Reasoning/LLMs-from-scratch-pyare/chapter-2-tokenization/2.9.png">

- We can modify the tokenizer to use an `<|unk|>` token if we encounter a word which is not part of the vocabulary.

- Further more we add the token `<|endoftext|>` token between unrealted texts.

<img src = "/DATA/pyare/Routine/LLM/Reasoning/LLMs-from-scratch-pyare/chapter-2-tokenization/2.10.png">

In [ ]:
all_tokens= sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>" ])
vocab = {token:integer for integer, token in enumerate(all_tokens)}

print(len(vocab.items())) # before it was 1130 and now it is 1132 vocab size.

In [ ]:
vocab.items()

In [ ]:
# for checking lets print the last 5 words of the updated vocabulary
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

In [ ]:
# Listing 2.4 A simple text tokenizer that handles unknown words.
class SimpleTokenizerV2:

    def __init__(self, vocab):

        self.str_to_int = vocab

        self.int_to_str = {i:s for s,i in vocab.items()}


    def encode(self,text):

        preprocessed = re.split(r'([,.:;?!"()\']|--|\s)', text) # this do the sepration of these from words

        preprocessed = [item.strip() for item in preprocessed if item.strip()] #Strip whitespace from each item and then filter out any empty strings.

        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed # if item is not in preprocessed then assign unk token to it
        ]

        ids = [self.str_to_int[s] for s in preprocessed]

        return ids

    def decode(self,ids):

        text = " ".join([self.int_to_str[i] for i in ids])
         # Replace spaces before the specified punctuations
        text = re.sub('\s+([,.:;?!"()\'])', r'\1',text) # The space before each punctuation mark gets removed while keeping the punctuation itself.

        return text



In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = "<|endoftext|>".join((text1,text2))

print(text)

In [ ]:
tokenizer.encode(text)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

---

- Some researcher also considered additional special tokens:
    -  [BOS](begninning of sequence)
    - [EOS](end of sequence ) : `<|endoftext|>`
    - [PAD] (padding) - when we train LLMs with batch sizes larger than one, the batch may contains the texts of varying lengths. To ensure all texts have the same length, the shorter texts are extended or "padded" using the [PAD] token, up to the length of the longest text in the batch. 

## 2.5 Byte pair Encoding


- We will do the BPE from scratch also but we could directly improt it from `tiktoken`

In [ ]:
!uv pip install tiktoken

In [ ]:
from importlib.metadata import version
import tiktoken

print("tiktoken version:", version("tiktoken"))

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")


In [ ]:
text = (
    "Hello, do you like tea?<|endoftext|>In the sunlit terrances"
    "of someunknownPlace."
)

integers = tokenizer.encode(text,allowed_special={"<|endoftext|>"}) # tokenID as 50256 for <|endoftext|>
print(integers)

In [ ]:
strings = tokenizer.decode(integers)
print(strings)

- BPE : correctly encode the unknown words without using the <|unk|> token, How does it do that?
- The Algorithm underlying the BPE breaks down words that aren't in its predefined vocabulary into smaller subword units or even individual characters, enabling it to handle out-of-vocabulary words.
- Thanks to the BPE algorithem, even when it encounters an unfimilar word during the tokenization, it can represent it as a sequence of subword tokens or characters, <br>

<img src =  "/DATA/pyare/Routine/LLM/Reasoning/LLMs-from-scratch-pyare/chapter-2-tokenization/2.11.png">

- The ability to break down unknown words into individaul characters ensure that the tokenizer and consequntly, the LLM that is trained with it can process any text. even if it contains words that were not present in the training data.

---

- [Orginal BPE tokenizer](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- [step wise implementation](https://github.com/rasbt/LLMs-from-scratch/tree/d85ba93799fcac69607ecf9fa554d22d0821fe20/ch02/05_bpe-from-scratch)
- Comparision: https://github.com/rasbt/LLMs-from-scratch/tree/d85ba93799fcac69607ecf9fa554d22d0821fe20/ch02/02_bonus_bytepair-encoder




There will be two implementations 1) simple which is easy for readability and 2) complex very similar to GPt-2 BPE implementation.
we will make the seprate notebooks for this 